# Análise de Liquidez — Ações Brasileiras (B3)

Este notebook analisa a **liquidez das ações negociadas na B3** a partir das
**cotações históricas oficiais do COTAHIST**, baixadas **diretamente da B3** — sem
depender de arquivos locais de terceiros.

## Sobre os dados

Fonte oficial: `https://bvmf.bmfbovespa.com.br/InstDados/SerHist/COTAHIST_A{ano}.ZIP`
(arquivos anuais da B3). O COTAHIST é um arquivo de **largura fixa (FWF)** em que cada
linha representa um negócio diário de um ativo; a primeira coluna indica o tipo de
registro (`01` = negócios, `00` = cabeçalho, `99` = trailer).

- Período analisado: **2025 em diante** (padrão: anos `2025` e `2026`, definidos em `ANOS`).
- Universo: ações à vista em **lote padrão** — filtro `instrument_market == '010'` e
  `bdi_code == '02'` com `specification_code` iniciando em `ON`/`PN`/`UNT`, o que
  inclui ações ON/PN e units (`11`) e exclui BDRs (`34`), FIIs e ETFs (ex.: BOVA11).
- Conversões: preços e volume financeiro vêm em **centavos** (÷100); o sentinela `-1`
  (sem dado) é tratado como `NaN`.
- Os arquivos anuais **não são acumulativos** (um registro por ativo/dia), portanto não
  há necessidade de deduplicação — aplicamos apenas uma checagem defensiva.
- Os downloads são armazenados em **cache local** (`../data/cotahist_acoes_*.csv.gz`)
  para evitar re-download nas execuções seguintes. Para atualizar os dados, basta apagar
  o arquivo de cache (ou incluir novos anos em `ANOS`).

> Observação: o layout do COTAHIST (posições das colunas) pode variar entre anos caso a
> B3 o altere; este notebook usa o layout padrão vigente para 2025/2026.


In [1]:
# Imports e configuração
import io
import os
import sys
import zipfile
import warnings

warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('..'))

import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from analise_liquidez.dados import ACOES_SCHEMA, ajustar_tipos

# Layout COTAHIST (largura fixa) — B3
URL_FORMAT = "https://bvmf.bmfbovespa.com.br/InstDados/SerHist/COTAHIST_A{ano}.ZIP"
COTAHIST_WIDTHS = [2, 8, 2, 12, 3, 12, 10, 3, 4, 13, 13, 13, 13, 13, 13, 13, 5, 18, 18, 13, 1, 8, 7, 13, 12, 3]
COTAHIST_FIELDS = [
    "regtype", "refdate", "bdi_code", "symbol", "instrument_market", "corporation_name", "specification_code",
    "days_to_settlement", "trading_currency", "open", "high", "low", "average", "close", "best_bid", "best_ask",
    "trade_quantity", "traded_contracts", "volume", "strike_price", "strike_price_adjustment_indicator",
    "maturity_date", "allocation_lot_size", "strike_price_in_points", "isin", "distribution_id",
]
HEADERS_HTTP = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept": "text/plain, */*",
    "Referer": "https://www.b3.com.br/",
    "Origin": "https://www.b3.com.br",
}


In [2]:
# Funções de download, parsing FWF e tratamento dos dados
def decode_bytes(content: bytes) -> str:
    """Decodifica o conteúdo bruto do COTAHIST (latin1/cp1252/utf-8)."""
    for enc in ("utf-8-sig", "latin1", "cp1252"):
        try:
            return content.decode(enc)
        except UnicodeDecodeError:
            continue
    return content.decode("utf-8", errors="replace")


def baixar_cotahist(ano: int) -> pd.DataFrame:
    """Baixa o COTAHIST anual da B3 e retorna os registros de negócios (regtype 01)."""
    url = URL_FORMAT.format(ano=ano)
    print(f"Baixando {url}")
    resp = requests.get(url, headers=HEADERS_HTTP, timeout=600)
    resp.raise_for_status()
    print(f"  {len(resp.content) / 1e6:.1f} MB recebidos")

    frames = []
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        for info in zf.infolist():
            if info.is_dir() or info.file_size == 0:
                continue
            text = decode_bytes(zf.read(info.filename))
            df = pd.read_fwf(
                io.StringIO(text),
                widths=COTAHIST_WIDTHS,
                names=COTAHIST_FIELDS,
                dtype=str,
                header=None,
                skiprows=1,  # pula o cabeçalho (regtype 00)
            )
            df = df[df["regtype"] == "01"].copy()
            print(f"  {info.filename}: {len(df):,} registros de negócios")
            frames.append(df)
    return pd.concat(frames, ignore_index=True)


def tratar_acoes(df: pd.DataFrame) -> pd.DataFrame:
    """Filtra ações à vista em lote padrão e converte tipos/preços."""
    df = df[(df["instrument_market"] == "010") & (df["bdi_code"] == "02")].copy()
    # Exclui ETFs/FIIs que negociam à vista: mantém apenas spec ON/PN/UNT (ações e units)
    spec = df["specification_code"].str.strip().str.upper()
    df = df[spec.str.startswith(("ON", "PN", "UNT"))].copy()
    df = df.rename(columns={
        "refdate": "data_referencia",
        "symbol": "codigo_ativo",
        "isin": "isin",
        "open": "preco_abertura",
        "high": "preco_maximo",
        "low": "preco_minimo",
        "average": "preco_medio",
        "close": "preco_ultimo",
        "trade_quantity": "trad_qty",
        "traded_contracts": "fin_instrm_qty",
        "volume": "ntl_fin_vol",
    })
    # Preços e volume financeiro vêm em centavos -> ÷100.
    # O sentinela "-1" (sem dado) vira NaN via to_numeric(errors='coerce').
    for col in ACOES_SCHEMA["price_cols"]:
        df[col] = pd.to_numeric(df[col], errors="coerce") / 100
    df["ntl_fin_vol"] = pd.to_numeric(df["ntl_fin_vol"], errors="coerce") / 100

    df = ajustar_tipos(df, ACOES_SCHEMA)
    colunas = ["data_referencia", "codigo_ativo", "isin"] + ACOES_SCHEMA["price_cols"] + \
              ACOES_SCHEMA["int_cols"] + ["ntl_fin_vol"]
    df = df[colunas]

    # Checagem defensiva: COTAHIST já é único por ativo/dia
    df = df.drop_duplicates(subset=["data_referencia", "codigo_ativo"], keep="last")
    return df.sort_values(["codigo_ativo", "data_referencia"]).reset_index(drop=True)


def carregar_cotahist(anos: list, cache_path: str) -> pd.DataFrame:
    """Baixa os anos solicitados (com cache local) e retorna as ações tratadas."""
    if os.path.exists(cache_path):
        print(f"Usando cache local: {cache_path}")
        df = pd.read_csv(cache_path, compression="gzip", dtype=str, low_memory=False)
        df = ajustar_tipos(df, ACOES_SCHEMA)
        df["ntl_fin_vol"] = pd.to_numeric(df["ntl_fin_vol"], errors="coerce").fillna(0.0)
        return df.sort_values(["codigo_ativo", "data_referencia"]).reset_index(drop=True)

    df_bruto = pd.concat([baixar_cotahist(a) for a in anos], ignore_index=True)
    df = tratar_acoes(df_bruto)
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    df.to_csv(cache_path, index=False, compression="gzip")
    print(f"Cache salvo em: {cache_path}")
    return df


In [3]:
# Download (primeira execução) ou cache local — anos 2025 e 2026
ANOS = [2025, 2026]
DIR_CACHE = os.path.join(os.path.abspath(".."), "data")
CACHE_PATH = os.path.join(DIR_CACHE, f"cotahist_acoes_{ANOS[0]}_{ANOS[-1]}.csv.gz")

df_acoes = carregar_cotahist(ANOS, CACHE_PATH)

print(f"Período: {df_acoes['data_referencia'].min().date()} a "
      f"{df_acoes['data_referencia'].max().date()}")
print(f"Tickers analisados: {df_acoes['codigo_ativo'].nunique():,}")
print(f"Linhas: {len(df_acoes):,} | Dias de negociação: {df_acoes['data_referencia'].nunique()}")
display(df_acoes.head())


Baixando https://bvmf.bmfbovespa.com.br/InstDados/SerHist/COTAHIST_A2025.ZIP


  89.1 MB recebidos


  COTAHIST_A2025.TXT: 3,174,698 registros de negócios
Baixando https://bvmf.bmfbovespa.com.br/InstDados/SerHist/COTAHIST_A2026.ZIP


  68.7 MB recebidos


  COTAHIST_A2026.TXT: 2,264,595 registros de negócios


Cache salvo em: /media/rodrigo/hd_1tb/projects/analise_liquidez_ativos/data/cotahist_acoes_2025_2026.csv.gz
Período: 2025-01-02 a 2026-07-31
Tickers analisados: 444
Linhas: 130,858 | Dias de negociação: 395


,data_referencia,codigo_ativo,isin,preco_abertura,preco_minimo,preco_maximo,preco_medio,preco_ultimo,trad_qty,fin_instrm_qty,ntl_fin_vol
0,2025-01-02,AALR3,BRAALRACNOR6,8.79,8.00,9.50,8.43,8.00,245,53000,447039.0
1,2025-01-03,AALR3,BRAALRACNOR6,8.14,7.27,8.15,7.56,7.27,284,66900,506392.0
2,2025-01-06,AALR3,BRAALRACNOR6,7.26,7.25,7.68,7.49,7.66,105,24200,181319.0
3,2025-01-07,AALR3,BRAALRACNOR6,7.66,7.40,8.00,7.73,7.90,59,12800,98945.0
4,2025-01-08,AALR3,BRAALRACNOR6,7.90,7.82,8.15,7.98,8.00,41,13800,110188.0


In [4]:
# Visão geral do mercado: volume financeiro, negócios e quantidade agregados por dia
print(f"Período: {df_acoes['data_referencia'].min().date()} a "
      f"{df_acoes['data_referencia'].max().date()}")
print(f"Tickers analisados: {df_acoes['codigo_ativo'].nunique():,}")
print(f"Dias de negociação: {df_acoes['data_referencia'].nunique()}")

# Cobertura diária de tickers (o arquivo do ano corrente é atualizado diariamente;
# o ranking ignora dias parciais)
cobertura_diaria = df_acoes.groupby('data_referencia')['codigo_ativo'].nunique()
cobertura_mediana = cobertura_diaria.median()
print(f"Cobertura mediana diária: {cobertura_mediana:.0f} tickers")

df_diario = (
    df_acoes.groupby('data_referencia', as_index=False)
    .agg(volume_financeiro=('ntl_fin_vol', 'sum'),
         negocios=('trad_qty', 'sum'),
         quantidade=('fin_instrm_qty', 'sum'))
)

fig_diario = px.line(
    df_diario, x='data_referencia', y='volume_financeiro',
    title='Volume Financeiro Diário Total — Ações (B3)',
    labels={'data_referencia': 'Data', 'volume_financeiro': 'Volume (R$)'}
)
fig_diario.update_yaxes(tickformat='~s')
fig_diario.show()


Período: 2025-01-02 a 2026-07-31
Tickers analisados: 444
Dias de negociação: 395
Cobertura mediana diária: 332 tickers


In [5]:
# Métricas de liquidez: médias móveis de 21 dias úteis por ativo
JANELA_DIAS = 21

def media_movel_por_ativo(serie):
    return serie.rolling(window=JANELA_DIAS, min_periods=1).mean()

df_acoes['volume_financeiro_media_21d'] = (
    df_acoes.groupby('codigo_ativo')['ntl_fin_vol'].transform(media_movel_por_ativo)
)
df_acoes['negocios_media_21d'] = (
    df_acoes.groupby('codigo_ativo')['trad_qty'].transform(media_movel_por_ativo)
)
df_acoes['quantidade_media_21d'] = (
    df_acoes.groupby('codigo_ativo')['fin_instrm_qty'].transform(media_movel_por_ativo)
)

print(f"Métricas de liquidez calculadas (janela de {JANELA_DIAS} dias úteis).")
display(df_acoes.tail())


Métricas de liquidez calculadas (janela de 21 dias úteis).


,data_referencia,codigo_ativo,isin,preco_abertura,preco_minimo,preco_maximo,preco_medio,preco_ultimo,trad_qty,fin_instrm_qty,ntl_fin_vol,volume_financeiro_media_21d,negocios_media_21d,quantidade_media_21d
130853,2025-11-26,ZAMP3,BRZAMPACNOR5,3.51,3.50,3.52,3.50,3.50,60,47500,166427.0,339598.095238,166.761905,94890.476190
130854,2025-11-27,ZAMP3,BRZAMPACNOR5,3.50,3.50,3.51,3.50,3.51,23,53600,187665.0,338215.047619,163.619048,94400.000000
130855,2025-11-28,ZAMP3,BRZAMPACNOR5,3.50,3.46,3.70,3.51,3.50,79,76900,270254.0,344615.761905,162.761905,96161.904762
130856,2025-12-01,ZAMP3,BRZAMPACNOR5,3.48,3.48,3.52,3.49,3.50,35,27200,95175.0,346551.571429,162.047619,96690.476190
130857,2025-12-02,ZAMP3,BRZAMPACNOR5,3.50,3.45,3.70,3.53,3.50,71,70200,247849.0,353062.619048,163.857143,98480.952381


In [6]:
# Ranking de liquidez na data mais recente com cobertura completa
# Dias recentes podem ter captura parcial; usamos o último dia com cobertura >= 90% da mediana
MIN_COBERTURA = 0.9

datas_completas = cobertura_diaria[cobertura_diaria >= cobertura_mediana * MIN_COBERTURA]
data_final = datas_completas.index.max()

print(f"Ranking calculado na data: {data_final.date()} "
      f"(cobertura de {cobertura_diaria[data_final]} tickers)")
print("\nCobertura dos últimos dias (dias parciais são ignorados no ranking):")
display(cobertura_diaria.tail(5).to_frame('nº de tickers'))

colunas_ranking = ['codigo_ativo', 'preco_medio', 'preco_ultimo',
                   'trad_qty', 'fin_instrm_qty', 'ntl_fin_vol',
                   'volume_financeiro_media_21d', 'negocios_media_21d']

df_ranking = (
    df_acoes[df_acoes['data_referencia'] == data_final]
    .sort_values('volume_financeiro_media_21d', ascending=False)
    [colunas_ranking]
    .rename(columns={'volume_financeiro_media_21d': 'volume_medio_21d',
                     'negocios_media_21d': 'negocios_medio_21d'})
)

display(df_ranking.head(20))


Ranking calculado na data: 2026-07-31 (cobertura de 312 tickers)

Cobertura dos últimos dias (dias parciais são ignorados no ranking):


,nº de tickers
data_referencia,
2026-07-27,318
2026-07-28,320
2026-07-29,313
2026-07-30,325
2026-07-31,312


,codigo_ativo,preco_medio,preco_ultimo,trad_qty,fin_instrm_qty,ntl_fin_vol,volume_medio_21d,negocios_medio_21d
89443,PETR4,43.36,43.42,45277,24997300,1.084119e+09,1.223390e+09,47256.190476
123324,VALE3,76.31,76.28,26634,17353300,1.324245e+09,1.129162e+09,30224.857143
67871,ITUB4,42.99,42.89,17315,10285400,4.421773e+08,7.342938e+08,29569.619048
9289,AXIA3,53.18,52.82,28460,9512100,5.058763e+08,5.497259e+08,24629.523810
13890,BBDC4,18.53,18.43,24451,25192900,4.669453e+08,5.065212e+08,26060.523810
11871,B3SA3,15.74,15.73,19790,24490400,3.857095e+08,4.780407e+08,28117.571429
1184,ABEV3,15.99,15.99,19324,38243300,6.115128e+08,4.505367e+08,21176.428571
20233,BPAC11,56.87,56.80,15260,5157000,2.933065e+08,4.309992e+08,20636.714286
89048,PETR3,49.08,49.04,10105,7065500,3.468008e+08,4.105522e+08,14302.095238
94047,PRIO3,60.76,60.85,16311,7178400,4.362254e+08,3.997138e+08,21505.571429


In [7]:
# Top 15 ações mais líquidas por volume financeiro médio de 21 dias
top15 = df_ranking.head(15).copy()
top15['rotulo'] = (
    top15['codigo_ativo'] + ' (R$ ' +
    top15['volume_medio_21d'].map(lambda v: f'{v/1e6:,.1f}M') + ')'
)

fig_top = px.bar(
    top15, x='rotulo', y='volume_medio_21d',
    title='Top 15 Ações por Volume Financeiro Médio de 21 Dias',
    labels={'rotulo': 'Ativo', 'volume_medio_21d': 'Volume Médio Diário (R$)'},
    color='volume_medio_21d', color_continuous_scale='Blues'
)
fig_top.update_layout(xaxis={'categoryorder': 'total descending'}, showlegend=False)
fig_top.update_yaxes(tickformat='~s')
fig_top.show()


In [8]:
# Análise individual de um ticker: preço médio + volume diário + volume médio 21d
ticker_alvo = 'PETR4'

df_ticker = df_acoes[df_acoes['codigo_ativo'] == ticker_alvo].copy()
print(f"Ticker: {ticker_alvo} | Dias com dados: {len(df_ticker)} | "
      f"Volume médio 21d: R$ {df_ticker['volume_financeiro_media_21d'].iloc[-1]:,.0f}")

fig_ind = go.Figure()
fig_ind.add_trace(go.Bar(x=df_ticker['data_referencia'], y=df_ticker['ntl_fin_vol'],
                         name='Volume Diário', yaxis='y2',
                         marker_color='rgba(120, 160, 220, 0.5)'))
fig_ind.add_trace(go.Scatter(x=df_ticker['data_referencia'], y=df_ticker['preco_medio'],
                             name='Preço Médio', mode='lines'))
fig_ind.add_trace(go.Scatter(x=df_ticker['data_referencia'],
                             y=df_ticker['volume_financeiro_media_21d'],
                             name='Volume Médio 21d', mode='lines', yaxis='y2',
                             line=dict(dash='dash', color='orange')))
fig_ind.update_layout(
    title=f'{ticker_alvo} — Preço Médio e Volume Financeiro Diário',
    xaxis_title='Data',
    yaxis_title='Preço Médio (R$)',
    yaxis2=dict(title='Volume Financeiro (R$)', overlaying='y', side='right'),
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0)
)
fig_ind.update_layout(yaxis2_tickformat='~s')
fig_ind.show()


Ticker: PETR4 | Dias com dados: 395 | Volume médio 21d: R$ 1,223,389,845


In [9]:
# Classificação qualitativa da liquidez por faixa de volume financeiro médio
limites_liquidez = [
    (50_000_000, 'Alta'),
    (10_000_000, 'Média'),
    (1_000_000, 'Baixa'),
    (0, 'Muito Baixa'),
]

def classificar_liquidez(volume_medio):
    for limite, classe in limites_liquidez:
        if volume_medio >= limite:
            return classe
    return 'Muito Baixa'

df_ranking['classificacao'] = df_ranking['volume_medio_21d'].apply(classificar_liquidez)

distribuicao = df_ranking['classificacao'].value_counts().reindex(
    ['Alta', 'Média', 'Baixa', 'Muito Baixa'])

print('Distribuição da classificação de liquidez:')
display(distribuicao.to_frame('nº de ativos'))

fig_class = px.bar(x=distribuicao.index, y=distribuicao.values,
                   title='Classificação de Liquidez das Ações',
                   labels={'x': 'Classificação', 'y': 'Nº de ativos'},
                   color=distribuicao.index)
fig_class.show()


Distribuição da classificação de liquidez:


,nº de ativos
classificacao,
Alta,64
Média,48
Baixa,80
Muito Baixa,120


In [10]:
# Análise de liquidez de carteira: dias necessários para liquidar posições
PARTICIPACAO_MAXIMA_DIARIA = 0.20  # até 20% do volume médio diário por dia

carteira = {
    'PETR4': 50_000,
    'VALE3': 80_000,
    'ITUB4': 60_000,
    'BBAS3': 90_000,
    'ABEV3': 120_000,
    'MGLU3': 100_000,
    'WEGE3': 30_000,
}

df_liq_ativo = df_ranking.set_index('codigo_ativo')

linhas = []
for ticker, quantidade in carteira.items():
    if ticker not in df_liq_ativo.index:
        linhas.append({'ticker': ticker, 'quantidade': quantidade, 'preco': np.nan,
                       'valor_posicao': np.nan, 'volume_medio_21d': np.nan,
                       'dias_para_liquidar': np.nan})
        continue

    preco = df_liq_ativo.at[ticker, 'preco_medio']
    volume_medio = df_liq_ativo.at[ticker, 'volume_medio_21d']
    valor_posicao = preco * quantidade
    capacidade_diaria = volume_medio * PARTICIPACAO_MAXIMA_DIARIA
    dias = np.ceil(valor_posicao / capacidade_diaria) if capacidade_diaria > 0 else np.nan

    linhas.append({'ticker': ticker, 'quantidade': quantidade, 'preco': preco,
                   'valor_posicao': valor_posicao, 'volume_medio_21d': volume_medio,
                   'dias_para_liquidar': dias})

df_carteira = pd.DataFrame(linhas)
pd.options.display.float_format = '{:,.2f}'.format
display(df_carteira)

fig_cart = px.bar(
    df_carteira.dropna(subset=['dias_para_liquidar']),
    x='ticker', y='dias_para_liquidar',
    title=f'Dias Necessários para Liquidar Posições '
          f'(participação de {PARTICIPACAO_MAXIMA_DIARIA:.0%} ao dia)',
    labels={'ticker': 'Ativo', 'dias_para_liquidar': 'Dias úteis'}
)
fig_cart.show()


,ticker,quantidade,preco,valor_posicao,volume_medio_21d,dias_para_liquidar
0,PETR4,50000,43.36,"2,168,000.00","1,223,389,845.43",1.00
1,VALE3,80000,76.31,"6,104,800.00","1,129,162,253.71",1.00
2,ITUB4,60000,42.99,"2,579,400.00","734,293,834.29",1.00
3,BBAS3,90000,21.21,"1,908,900.00","376,891,584.95",1.00
4,ABEV3,120000,15.99,"1,918,800.00","450,536,687.33",1.00
5,MGLU3,100000,5.02,"502,000.00","86,574,141.10",1.00
6,WEGE3,30000,47.33,"1,419,900.00","379,044,270.05",1.00


In [11]:
# Exportação opcional do ranking consolidado para Excel
caminho_excel = 'relatorio_liquidez_acoes.xlsx'
df_ranking.to_excel(caminho_excel, sheet_name='Liquidez Acoes', index=False)
print(f"Ranking exportado para {caminho_excel}")


Ranking exportado para relatorio_liquidez_acoes.xlsx
